# JAXTPC Pixel Simulation

GPU-accelerated pixel TPC detector simulation.

**Workflow:**
1. Load the pixel detector configuration and generate a synthetic multi-track event inline
2. Run simulation (recombination, drift, diffusion, pixel response)
3. Convert output to sparse format
4. Visualize: pixel projections, anode heatmap, waveforms

**Note:** the production pixel config is 1000×1000 (~10.8 GB dense per volume —
needs a large GPU). This notebook downsamples the grid (`DEMO_PIXELS`) so it
runs on modest hardware; set `DEMO_PIXELS = None` for full resolution. The event
is generated inline — swap `make_synthetic_event(...)` for
`load_event(path, cfg, event_idx=...)` to simulate real data.

## Setup and Configuration

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

CONFIG_PATH = "config/cubic_pixel_config.yaml"

# Synthetic event (generated inline — no external data file needed)
SEED = 0
N_TRACKS = 6
STEP_CM = 0.4

# Pixel grid for this interactive demo. The production config is 1000x1000
# (~10.8 GB dense per volume). We downsample the grid here, keeping the same
# physical coverage, so the notebook runs on modest hardware.
# Set DEMO_PIXELS = None to use the config's full resolution (needs a large GPU).
DEMO_PIXELS = 320

# Threshold for sparse output (electrons)
THRESHOLD_ENC = 500

# Performance knobs
TOTAL_PAD = 50_000

print("Configuration:")
print(f"  Config: {CONFIG_PATH}")
print(f"  Synthetic event: {N_TRACKS} tracks (seed={SEED})")
print(f"  Demo pixel grid: {DEMO_PIXELS or 'full (from config)'}")
print(f"  Threshold: {THRESHOLD_ENC} e-")
print(f"  total_pad: {TOTAL_PAD:,}")

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import os
import time
import gc

from tools.simulation import DetectorSimulator
from tools.geometry import generate_detector
from tools.loader import build_deposit_data   # load_event(path, cfg, event_idx=...) for real data
from tools.output import to_sparse

from tools.pixel_visualization import (
    visualize_pixel_projections,
    visualize_pixel_anode,
    visualize_pixel_waveforms,
)

os.makedirs("plots", exist_ok=True)

print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

## Build Simulator and Synthetic Event

In [ ]:
# =============================================================================
# BUILD SIMULATOR + SYNTHETIC EVENT
# =============================================================================

detector_config = generate_detector(CONFIG_PATH)

# Downsample the pixel grid for this demo (keep physical coverage constant).
if DEMO_PIXELS is not None:
    for vol in detector_config['volumes']:
        ro = vol['readout']
        extent_cm = ro['pixel_shape'][0] * ro['pixel_pitch']
        ro['pixel_shape'] = [DEMO_PIXELS, DEMO_PIXELS]
        ro['pixel_pitch'] = extent_cm / DEMO_PIXELS

jax.clear_caches()
gc.collect()

simulator = DetectorSimulator(
    detector_config,
    total_pad=TOTAL_PAD,
)

cfg = simulator.config

print(f"Volumes: {cfg.n_volumes}")
for v in range(cfg.n_volumes):
    vol = cfg.volumes[v]
    print(f"  Volume {v}: {vol.readout_type}, pixel_shape={vol.pixel_shape}, "
          f"pitch={vol.pixel_pitch_cm*10:.2f}mm")


def make_synthetic_event(seed=0, n_tracks=6, step_cm=0.4):
    """A few straight MIP-like tracks crossing the dual-TPC detector.

    Returns global-frame (positions_mm, de, dx, theta, phi, track_ids) — the
    same arrays an edepsim loader would provide. To simulate real data instead,
    replace the call below with ``load_event(path, cfg, event_idx=...)``.
    """
    rng = np.random.RandomState(seed)
    P, DE, DX, TH, PH, TID = [], [], [], [], [], []
    for t in range(n_tracks):
        start = rng.uniform(-180, 180, size=3)
        d = rng.normal(size=3); d /= np.linalg.norm(d)
        n_seg = int(rng.uniform(80, 250) / step_cm)
        s = np.arange(n_seg) * step_cm
        pts = np.clip(start[None, :] + s[:, None] * d[None, :], -215.9, 215.9)
        dedx = rng.uniform(1.8, 2.6)  # MeV/cm, MIP-like
        P.append(pts)
        DE.append(np.full(n_seg, dedx * step_cm, np.float32))
        DX.append(np.full(n_seg, step_cm, np.float32))
        TH.append(np.full(n_seg, np.arccos(np.clip(d[0], -1, 1)), np.float32))
        PH.append(np.full(n_seg, np.arctan2(d[2], d[1]), np.float32))
        TID.append(np.full(n_seg, t, np.int32))
    return ((np.concatenate(P) * 10.0).astype(np.float32),
            np.concatenate(DE), np.concatenate(DX),
            np.concatenate(TH), np.concatenate(PH), np.concatenate(TID))


pos_mm, de, dx, theta, phi, track_ids = make_synthetic_event(
    seed=SEED, n_tracks=N_TRACKS, step_cm=STEP_CM)
deposits = build_deposit_data(
    pos_mm, de, dx, cfg, theta=theta, phi=phi, track_ids=track_ids)

n_volumes = len(deposits.volumes)
n_total = sum(v.n_actual for v in deposits.volumes)
print(f"\nGenerated {n_total:,} deposits across {n_volumes} volumes")
for v in range(n_volumes):
    print(f"  Volume {v}: {deposits.volumes[v].n_actual:,} deposits")

## Run Simulation

In [ ]:
# =============================================================================
# WARM UP
# =============================================================================

print("Warming up JIT...")
simulator.warm_up()
print("Done.")

In [ ]:
# =============================================================================
# RUN SIMULATION
# =============================================================================

print("Running simulation...")
t0 = time.time()

response_signals, track_hits_raw, deposits = simulator.process_event(
    deposits, key=jax.random.PRNGKey(42))

# Wait for GPU
for val in response_signals.values():
    if isinstance(val, dict):
        for a in val.values():
            jax.block_until_ready(a)
    elif isinstance(val, tuple):
        jax.block_until_ready(val[0])
    else:
        jax.block_until_ready(val)

elapsed = time.time() - t0
print(f"Simulation: {elapsed:.2f}s ({n_total / elapsed:,.0f} segments/sec)")

# Show signal stats
for (vi, pi), sig in response_signals.items():
    if isinstance(sig, dict):
        n = len(sig['values'])
        n_neg = np.sum(np.asarray(sig['values']) < 0)
        print(f"  Volume {vi}: {n:,} signal entries ({n_neg:,} negative)")

# Show track hits stats
for k, v in track_hits_raw.items():
    if isinstance(k, tuple):
        print(f"  Track hits {k}: {int(v[4]):,} entries")

## Convert to Sparse

In [ ]:
# =============================================================================
# CONVERT TO SPARSE
# =============================================================================

threshold_adc = THRESHOLD_ENC / cfg.electrons_per_adc
t0 = time.time()
sparse_signals = to_sparse(response_signals, cfg, threshold_adc=threshold_adc)
print(f"to_sparse ({THRESHOLD_ENC} e-): {time.time() - t0:.2f}s")

print(f"\nSparse output:")
for (vi, pi), data in sorted(sparse_signals.items()):
    n_vox = len(data['values'])
    total = cfg.volumes[vi].pixel_shape[0] * cfg.volumes[vi].pixel_shape[1] * cfg.num_time_steps
    sparsity = (1 - n_vox / total) * 100
    print(f"  Volume {vi}: {n_vox:,} voxels ({sparsity:.3f}% sparse)")

## Visualize Projections

In [ ]:
# =============================================================================
# THREE ORTHOGONAL PROJECTIONS
# =============================================================================

for v in range(n_volumes):
    fig = visualize_pixel_projections(
        sparse_signals, cfg, vol_idx=v,
        log_norm=True, threshold=threshold_adc, reduce='sum')
    fig.savefig(f"plots/pixel_projections_vol{v}.png", dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()

## Anode Heatmap

In [ ]:
# =============================================================================
# PIXEL ANODE HEATMAP (Y-Z plane, summed over time)
# =============================================================================

for v in range(n_volumes):
    fig = visualize_pixel_anode(
        sparse_signals, cfg, vol_idx=v,
        log_norm=True, threshold=threshold_adc)
    fig.savefig(f"plots/pixel_anode_vol{v}.png", dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()

## Pixel Waveforms

In [ ]:
# =============================================================================
# WAVEFORMS AT SPECIFIC PIXELS
# =============================================================================

# Find the hottest pixels in volume 0
key = (0, 0)
if key in sparse_signals and len(sparse_signals[key]['values']) > 0:
    data = sparse_signals[key]
    py, pz, vals = data['pixel_y'], data['pixel_z'], data['values']
    
    # Aggregate charge per pixel
    pixel_charge = {}
    for i in range(len(vals)):
        k = (int(py[i]), int(pz[i]))
        pixel_charge[k] = pixel_charge.get(k, 0) + abs(float(vals[i]))
    
    top_pixels = sorted(pixel_charge.items(), key=lambda x: x[1], reverse=True)[:6]
    coords = [k for k, _ in top_pixels]
    
    print("Top pixels by charge:")
    for (py_i, pz_i), ch in top_pixels:
        print(f"  ({py_i}, {pz_i}): {ch:,.1f}")
    
    fig = visualize_pixel_waveforms(sparse_signals, cfg, pixel_coords=coords, vol_idx=0)
    fig.savefig("plots/pixel_waveforms.png", dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()
else:
    print("No signal in volume 0.")

## Summary

In [ ]:
# =============================================================================
# SUMMARY
# =============================================================================

print("=" * 60)
print(" PIXEL SIMULATION SUMMARY")
print("=" * 60)

print(f"\nInput:")
print(f"  Synthetic event: {N_TRACKS} tracks (seed={SEED})")
print(f"  Total deposits: {n_total:,}")
for v in range(n_volumes):
    print(f"  Volume {v}: {deposits.volumes[v].n_actual:,}")

print(f"\nDetector:")
for v in range(n_volumes):
    vol = cfg.volumes[v]
    print(f"  Volume {v}: {vol.pixel_shape[0]}x{vol.pixel_shape[1]} pixels, "
          f"{vol.pixel_pitch_cm*10:.2f}mm pitch")

print(f"\nSimulation:")
print(f"  Time: {elapsed:.2f}s")
print(f"  Throughput: {n_total / elapsed:,.0f} seg/s")

total_vox = sum(len(d['values']) for d in sparse_signals.values())
total_possible = sum(
    cfg.volumes[v].pixel_shape[0] * cfg.volumes[v].pixel_shape[1] * cfg.num_time_steps
    for v in range(cfg.n_volumes)
)
print(f"\nSparse output ({THRESHOLD_ENC} e-):")
print(f"  Voxels: {total_vox:,}")
print(f"  Sparsity: {(1 - total_vox/total_possible)*100:.3f}%")

# Track hits summary
for k, v in track_hits_raw.items():
    if isinstance(k, tuple):
        count = int(v[4])
        print(f"  Track hits {k}: {count:,} entries")